# Foundations, Build & Conformance Harness — epic proof

Runs `services/foundations-reference-architecture.ts` (the module the cucumber steps drive): the workspace skeleton validates with adapter boundary lint enforced, and hello-bindle componentizes deterministically down to the OCI descriptor and smoke run.

In [1]:
import { strict as assert } from "node:assert";

import { runFoundationsBuildConformanceSimulation } from "./services/foundations-reference-architecture.ts";

const result = await runFoundationsBuildConformanceSimulation();
assert.equal(result.epic.slug, "foundations-build-conformance-harness");

// 1) Workspace skeleton validates; adapter boundary violations are denied.
assert.equal(result.workspace.packageManager, "pnpm");
assert.equal(result.workspace.status, "valid");
assert.equal(result.workspace.boundaryViolation.effect, "deny");
assert.equal(result.negative.adapterImport.effect, "deny");
assert.match(
	result.negative.adapterImport.reason,
	/adapters may import only core ports/,
);
console.log(
	`workspace status=${result.workspace.status} packageManager=${result.workspace.packageManager} boundaryViolation=${result.workspace.boundaryViolation.effect}`,
);
console.log(
	`adapter import lint effect=${result.negative.adapterImport.effect}`,
);

// 2) hello-bindle componentizes deterministically: wasm validates, OCI descriptor, smoke run.
const componentize = result.componentize;
assert.equal(componentize.validation.passed, true);
assert.equal(componentize.smoke.exitCode, 0);
assert.equal(
	componentize.oci.mediaType,
	"application/vnd.oci.image.manifest.v1+json",
);
console.log(
	`componentize wasmValid=${componentize.validation.passed} smokeExit=${componentize.smoke.exitCode} ociMediaType=${componentize.oci.mediaType}`,
);

workspace status=valid packageManager=pnpm boundaryViolation=deny
adapter import lint effect=deny
componentize wasmValid=true smokeExit=0 ociMediaType=application/vnd.oci.image.manifest.v1+json


Trust and CI: signed artifacts verify into Matching without being auto-selected, unsigned artifacts are refused, and the E2E lane runs only on short-lived OIDC tokens — no stored secrets, missing OIDC fails closed.

In [2]:
// 3) Trust gate: signed artifacts verify and reach Matching without auto-selection;
//    unsigned artifacts are refused.
assert.equal(result.artifactTrust.install.ok, true);
assert.equal(result.artifactTrust.install.availableToMatching, true);
assert.equal(result.artifactTrust.install.selectedByMatching, false);
assert.deepEqual(result.negative.unsignedArtifact.reasons, [
	"signature.required",
]);
console.log(
	`trust gate installOk=${result.artifactTrust.install.ok} availableToMatching=${result.artifactTrust.install.availableToMatching} autoSelected=${result.artifactTrust.install.selectedByMatching}`,
);
console.log(
	`unsigned artifact refused reasons=${result.negative.unsignedArtifact.reasons.join(",")}`,
);

// 4) CI: contract lane passes; E2E runs on short-lived OIDC tokens; missing OIDC fails closed.
assert.equal(result.ci.contract.passed, true);
assert.equal(result.ci.e2e.status, "passed");
assert.equal(result.ci.e2e.token.expiresInSeconds, 900);
assert.equal(result.negative.ciMissingOidc.passed, false);
console.log(
	`ci contract=${result.ci.contract.passed} e2e=${result.ci.e2e.status} tokenTtl=${result.ci.e2e.token.expiresInSeconds}s missingOidcPasses=${result.negative.ciMissingOidc.passed}`,
);

console.log(
	"EPIC HELD: skeleton lint, deterministic builds, signed-artifact trust, and OIDC-gated CI all enforce",
);

trust gate installOk=true availableToMatching=true autoSelected=false
unsigned artifact refused reasons=signature.required
ci contract=true e2e=passed tokenTtl=900s missingOidcPasses=false
EPIC HELD: skeleton lint, deterministic builds, signed-artifact trust, and OIDC-gated CI all enforce


Every assertion above ran green — the same invariants the executable `features/*.feature` scenarios enforce in the repo cucumber lane. `examples/usage.ts` is the gate-executed twin of this notebook.